# Lab 3: Measuring Networks & Random Graph Models
**Course:** IS353 - Social Network Analysis, UIT, VNUHCM  
**Lecturer:** Dr. Tran Hung Nghiep, 2025  

---

**Goals:**
- Compute degree distribution, clustering, path length, diameter, connected components
- Generate and measure **Erdős–Rényi** $G_{n,p}$ and **Watts–Strogatz** small-world graphs; compare real graph and toy graph
- (Bonus) Configuration model with preserved degree sequence


In [ ]:

# Setup
import os, math, random, time
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

def path_metrics_on_lcc(G):
    import numpy as np
    if G.number_of_nodes() == 0:
        return np.nan, np.nan, 0
    if G.is_directed():
        if nx.is_strongly_connected(G):
            H = G
        else:
            sccs = sorted(nx.strongly_connected_components(G), key=len, reverse=True)
            H = G.subgraph(sccs[0]).copy()
    else:
        if nx.is_connected(G):
            H = G
        else:
            ccs = sorted(nx.connected_components(G), key=len, reverse=True)
            H = G.subgraph(ccs[0]).copy()
    try:
        asp = nx.average_shortest_path_length(H)
        dia = nx.diameter(H)
    except Exception:
        asp, dia = np.nan, np.nan
    return asp, dia, H.number_of_nodes()

def degree_histogram(G, bins=50, title="Degree distribution"):
    degs = [d for _, d in G.degree()]
    plt.figure(figsize=(6,4))
    plt.hist(degs, bins=bins)
    plt.xlabel("Degree"); plt.ylabel("Frequency"); plt.title(title)
    plt.tight_layout()
    plt.show()
    return degs



## 1) Load a real/toy graph
- Use your `toy_edges.txt` from Lab 1, **or** replace the path with any small real dataset (e.g., `facebook_combined.txt` after subsampling).
- For path metrics we compute on the **largest connected component (LCC)**.


In [ ]:

# Change this path as needed
path = "/content/toy_edges.txt"

def load_or_make_toy(path):
    if not os.path.exists(path):
        edges = [
            ("Alice","Bob"),("Alice","Carol"),("Bob","Carol"),("Carol","Dave"),("Bob","Dave"),
            ("Eve","Frank"),("Eve","Grace"),("Frank","Grace"),("Grace","Heidi"),("Frank","Heidi"),
            ("Carol","Eve"),("Dave","Frank"),("Heidi","Ivan"),("Ivan","Judy"),("Judy","Grace"),("Alice","Ivan"),
        ]
        with open(path, "w", encoding="utf-8") as f:
            for u,v in edges:
                f.write(f"{u} {v}\n")
    G = nx.read_edgelist(path, create_using=nx.Graph)
    return G

G_real = load_or_make_toy(path)
print("Real/Toy graph:", G_real.number_of_nodes(), "nodes,", G_real.number_of_edges(), "edges")

density = nx.density(G_real)
avg_clust = nx.average_clustering(G_real)
asp, dia, lcc_n = path_metrics_on_lcc(G_real)

print(f"Density={density:.4f} | Avg clustering={avg_clust:.4f} | ASP(LCC)={asp} | Diameter(LCC)={dia} | LCC size={lcc_n}")
degs = degree_histogram(G_real, title="Degree distribution (real/toy)")

if nx.is_connected(G_real):
    G_base = G_real.copy()
else:
    cc = max(nx.connected_components(G_real), key=len)
    G_base = G_real.subgraph(cc).copy()
n_base = G_base.number_of_nodes()
m_base = G_base.number_of_edges()
avg_deg_base = 2*m_base/n_base if n_base>0 else 0
print("Base (LCC) n,m,avg_deg:", n_base, m_base, f"{avg_deg_base:.2f}")



## 2) Erdős–Rényi Random Graph $G_{n,p}$
We match $n$ and the expected average degree by $p = \frac{2m}{n(n-1)}$.


In [ ]:

if n_base >= 2:
    p = (2*m_base) / (n_base*(n_base-1))
else:
    p = 0.1
G_gnp = nx.gnp_random_graph(n_base, p, seed=42)

density_gnp = nx.density(G_gnp)
avg_clust_gnp = nx.average_clustering(G_gnp)
asp_gnp, dia_gnp, lcc_n_gnp = path_metrics_on_lcc(G_gnp)

print("Gnp:", G_gnp.number_of_nodes(), "nodes,", G_gnp.number_of_edges(), "edges")
print(f"p={p:.6f} | Density={density_gnp:.4f} | Avg clustering={avg_clust_gnp:.4f} | ASP(LCC)={asp_gnp} | Diameter(LCC)={dia_gnp} | LCC size={lcc_n_gnp}")
degree_histogram(G_gnp, title="Degree distribution (Gnp)")



## 3) Watts–Strogatz Small-World Graph
Choose $k \approx$ average degree (nearest **even**), with a small rewiring probability.


In [ ]:

if n_base < 4:
    n_ws, k_ws, p_rewire = max(n_base, 4), 2, 0.1
else:
    k_ws = max(2, int(round(avg_deg_base)))
    if k_ws % 2 == 1:
        k_ws += 1
    n_ws = n_base
    p_rewire = 0.1

G_ws = nx.watts_strogatz_graph(n_ws, k_ws, p_rewire, seed=42)

density_ws = nx.density(G_ws)
avg_clust_ws = nx.average_clustering(G_ws)
asp_ws, dia_ws, lcc_n_ws = path_metrics_on_lcc(G_ws)

print("WS:", G_ws.number_of_nodes(), "nodes,", G_ws.number_of_edges(), "edges")
print(f"k={k_ws} p_rewire={p_rewire} | Density={density_ws:.4f} | Avg clustering={avg_clust_ws:.4f} | ASP(LCC)={asp_ws} | Diameter(LCC)={dia_ws} | LCC size={lcc_n_ws}")
degree_histogram(G_ws, title="Degree distribution (Watts–Strogatz)")



## 4) Compare metrics


In [ ]:

import pandas as pd
rows = [
    ["Real/LCC", n_base, G_base.number_of_edges(), nx.density(G_base), nx.average_clustering(G_base), *path_metrics_on_lcc(G_base)[:2]],
    ["Gnp", G_gnp.number_of_nodes(), G_gnp.number_of_edges(), density_gnp, avg_clust_gnp, asp_gnp, dia_gnp],
    ["Watts–Strogatz", G_ws.number_of_nodes(), G_ws.number_of_edges(), density_ws, avg_clust_ws, asp_ws, dia_ws],
]
df = pd.DataFrame(rows, columns=["Graph","n","m","Density","AvgClustering","AvgShortestPath(LCC)","Diameter(LCC)"])
df



## 5) (Bonus) Configuration Model: preserve degree sequence


In [ ]:

deg_seq = [d for _, d in G_base.degree()]
G_conf_multi = nx.configuration_model(deg_seq, seed=42)
G_conf = nx.Graph(G_conf_multi)
G_conf.remove_edges_from(nx.selfloop_edges(G_conf))

density_conf = nx.density(G_conf)
avg_clust_conf = nx.average_clustering(G_conf)
asp_conf, dia_conf, lcc_n_conf = path_metrics_on_lcc(G_conf)

print("Configuration model (simple):", G_conf.number_of_nodes(), "nodes,", G_conf.number_of_edges(), "edges")
print(f"Density={density_conf:.4f} | Avg clustering={avg_clust_conf:.4f} | ASP(LCC)={asp_conf} | Diameter(LCC)={dia_conf} | LCC size={lcc_n_conf}")
degree_histogram(G_conf, title="Degree distribution (Configuration model)")


## Questions:
1. Compute degree distribution, clustering, path length, diameter, connected components of the ego-Facebook networks.

2. Generate and measure the Gn,p and WS graphs that has similar number of nodes and edges to the ego-Facebook networks.

3. Compare the measurements of the ego-Facebook networks and the corresponding Gn,p and WS graphs:
   - Present in a table format
   - Plot using linear-linear plot and log-log plot
   - What models best fit the ego-Facebook networks?
   - What can we conclude from the measurements?